<a href="https://colab.research.google.com/github/Integenpiyush/churn-prediction-catboost/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install catboost optuna shap wandb imbalanced-learn -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [14]:
df = pd.read_csv('/content/telco_cleaned.csv')
print("Shape:", df.shape)
print("Churn distribution:")
print(df['Churn'].value_counts(normalize=True).round(3) * 100)

Shape: (7032, 20)
Churn distribution:
Churn
0    73.4
1    26.6
Name: proportion, dtype: float64


## Preprocessing Plan
1. Drop TotalCharges — high multicollinearity (0.83 with tenure)
2. Encode binary categorical columns with LabelEncoder
3. Encode multi-class categorical columns with get_dummies
4. Train-test split FIRST (80/20)
5. Scale numerical columns — fit on train only, transform both
6. Apply SMOTE on training data only — never on test

In [15]:
#  Drop TotalCharges
df = df.drop(columns=['TotalCharges'])
print("TotalCharges dropped. Remaining columns:", df.shape[1])
print(df.columns.tolist())

TotalCharges dropped. Remaining columns: 19
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'Churn']


In [16]:
# Encode binary categorical columns
binary_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
               'PhoneService', 'PaperlessBilling']
le = LabelEncoder()
for col in binary_cols:
  df[col] = le.fit_transform(df[col])
  print(f"{col}: {df[col].unique()}")

gender: [0 1]
SeniorCitizen: [0 1]
Partner: [1 0]
Dependents: [0 1]
PhoneService: [0 1]
PaperlessBilling: [1 0]


In [17]:
# Encode multi-class categorical columns
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract',
              'PaymentMethod']

df = pd.get_dummies(df,columns=multi_cols,drop_first = True)
print("Shape after encoding:", df.shape)
print("New columns added:", df.shape[1])

Shape after encoding: (7032, 30)
New columns added: 30


In [18]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,Churn,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,0,True,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,0,False,...,False,False,False,False,False,True,False,False,False,True
2,1,0,0,0,2,1,1,53.85,1,False,...,False,False,False,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,0,True,...,True,False,False,False,False,True,False,False,False,False
4,0,0,0,0,2,1,1,70.70,1,False,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,1,0,1,1,24,1,1,84.80,0,False,...,True,False,True,False,True,True,False,False,False,True
7028,0,0,1,1,72,1,1,103.20,0,False,...,False,False,True,False,True,True,False,True,False,False
7029,0,0,1,1,11,0,1,29.60,0,True,...,False,False,False,False,False,False,False,False,True,False
7030,1,1,1,0,4,1,1,74.40,1,False,...,False,False,False,False,False,False,False,False,False,True


## Encoding Decisions
- Binary columns (Yes/No): LabelEncoder → 0 and 1
  Simple, no extra columns created
- Multi-class columns: pd.get_dummies with drop_first=True
  drop_first avoids dummy variable trap —
  if you have 3 categories A/B/C and know A=0, B=0, then C=1 is implied.
  Keeping all 3 dummies creates perfect multicollinearity.

In [19]:
# Separate features and target
X = df.drop(columns = ['Churn'])
y = df['Churn']
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:")
print(y.value_counts())

Features shape: (7032, 29)
Target shape: (7032,)
Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [20]:
# train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    random_state=42,
    test_size = 0.2,
    stratify = y # maintains class ratio in both splits
    )
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain churn distribution:")
print(y_train.value_counts(normalize=True).round(3) * 100)
print("\nTest churn distribution:")
print(y_test.value_counts(normalize=True).round(3) * 100)

Train size: (5625, 29)
Test size: (1407, 29)

Train churn distribution:
Churn
0    73.4
1    26.6
Name: proportion, dtype: float64

Test churn distribution:
Churn
0    73.4
1    26.6
Name: proportion, dtype: float64


## Why stratify=y in train_test_split?
Without stratify, random splitting might put most churned customers
in train and very few in test — making evaluation unreliable.
stratify=y ensures both splits maintain the original 73.5/26.5
class ratio. Always use stratify for imbalanced classification problems.

## Train-Test Split Result
- Train: 5,625 rows | Test: 1,407 rows (80/20 split)
- Train churn: 73.4% No, 26.6% Yes
- Test churn: 73.4% No, 26.6% Yes
- stratify=y confirmed working — both splits maintain original distribution
- Test set will NOT be touched until final model evaluation

In [21]:
#  Scale numerical columns
scaler = StandardScaler()
numerical_cols = ['tenure','MonthlyCharges']
X_train[numerical_cols] = scaler.fit_transform( X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform( X_test[numerical_cols])

print("Scaling complete.")
print("\nTrain tenure stats after scaling:")
print(X_train['tenure'].describe().round(3))


Scaling complete.

Train tenure stats after scaling:
count    5625.000
mean       -0.000
std         1.000
min        -1.286
25%        -0.960
50%        -0.145
75%         0.955
max         1.607
Name: tenure, dtype: float64


## Why fit scaler on train only?
scaler.fit_transform(X_train) — learns mean and std FROM training data only
scaler.transform(X_test)      — applies those same values to test data

If we fit on full dataset:
- Test set statistics leak into training
- Model sees future data during training
- Performance is artificially inflated
- In production the scaler will never see future data

This is called data leakage — one of the most common mistakes
in ML projects.

In [22]:
print("Before SMOTE:")
print("X_train shape:", X_train.shape)
print(y_train.value_counts())


smote = SMOTE(random_state = 42)
X_train_smote,y_train_smote = smote.fit_resample(X_train,y_train)


print("\nAfter SMOTE:")
print("X_train shape:", X_train_smote.shape)
print(y_train_smote.value_counts())

Before SMOTE:
X_train shape: (5625, 29)
Churn
0    4130
1    1495
Name: count, dtype: int64

After SMOTE:
X_train shape: (8260, 29)
Churn
0    4130
1    4130
Name: count, dtype: int64


In [23]:
# Save preprocessed data
import pickle

pickle.dump((X_train_smote,X_test,y_train_smote ,y_test),open('preprocessed_data.pkl','wb'))

pickle.dump(scaler, open('scaler.pkl', 'wb'))

print("Saved:")
print("- preprocessed_data.pkl")
print("- scaler.pkl")
print("\nFinal shapes going into modeling:")
print("X_train (after SMOTE):", X_train_smote.shape)
print("X_test:", X_test.shape)
print("y_train (after SMOTE):", y_train_smote.value_counts().to_dict())
print("y_test:", y_test.value_counts().to_dict())

Saved:
- preprocessed_data.pkl
- scaler.pkl

Final shapes going into modeling:
X_train (after SMOTE): (8260, 29)
X_test: (1407, 29)
y_train (after SMOTE): {0: 4130, 1: 4130}
y_test: {0: 1033, 1: 374}


## Preprocessing Complete

### Scaling verification:
- tenure mean after scaling: 0.000 ✓
- tenure std after scaling: 1.000 ✓
- StandardScaler working correctly

### SMOTE result:
- Before: 4,130 No churn vs 1,495 churn (imbalanced)
- After: 4,130 No churn vs 4,130 churn (perfectly balanced)
- Synthetic samples created: 2,635 new minority samples
- Training set grew from 5,625 → 8,260 rows

### What goes into modeling:
- X_train: 8,260 rows, 29 features (SMOTE applied)
- X_test: 1,407 rows, 29 features (original distribution kept)
- Test set untouched — reflects real world 73.4/26.6 distribution

"What did SMOTE actually do to your training data?"
Your answer: "SMOTE took the 1,495 churned customers in training, found nearest neighbors for each in feature space, and generated 2,635 synthetic churned customers between them — bringing the minority class from 1,495 to 4,130, matching the majority class exactly."
"Why is your test set 1,407 rows and not balanced?"
Your answer: "The test set represents real world distribution — in reality 73.4% of customers don't churn. Balancing the test set would mean evaluating on an artificial scenario that doesn't exist in production. I kept it at original distribution so my metrics reflect genuine real-world performance."